# Lab: The Decoding Bench

**Bluevision-ai Academy**
*GPT: Predicting What Comes Next*

Session 07 ended on a gap: the model hands you a distribution over the vocabulary, and something else has to turn that distribution into an actual token. This lab is that something else. You build a small bench that runs a real GPT-2 under every decoding rule the lecture named, put a number on what each rule actually buys and costs, then use those numbers to answer a question the slides only posed in words: which setting for which job?

Each part is a hands-on build: you start from the problem, make the design choice, then read the result off your own numbers. Five moves structure each part: **Problem** (what we have, what we want) -> **Ingredients** (what the build needs, nothing else yet) -> **Idea** (the design choice, in words, before any code) -> **Build** (the code that embodies exactly that idea) -> **Payoff** (what the result means, plus a limitation). One idea per cell -- if a cell needs "and" to describe it, it is doing too much.

| Part | What you build | Builds on (lecture) |
|---|---|---|
| 1 -- Build the Bench | A harness that runs GPT-2 under greedy, temperature, top-k and nucleus decoding | Slides 21, 24 (temperature, nucleus) |
| 2 -- Measure the Trade-off | Two real metrics -- repetition rate, mean log-probability -- scored over every run | Slide 23 (the quiz's own numbers), Slide 24 (Holtzman et al.) |
| 3 -- Few-shot, and Where It Stops | The shot count where in-context learning starts working, and a phrasing where it breaks | Slides 18-19 (prompt-as-program, in-context learning) |
| Challenge | Break it on purpose: one default setting, two jobs | Slides 21-24, applied to a decision |
| Reflection | Five questions, answered in code comments | -- |

**Model:** `gpt2` (124M, the original OpenAI checkpoint) -- small enough to run every cell on a Colab CPU runtime in a few minutes, and the same architecture family the lecture's `gpt_block` and `gpt_stack` figures describe. **Estimated time:** ~2 hours. Runs top to bottom on Google Colab with no setup beyond the cell below.

## Setup

Colab already ships with most of this. The cell below pins versions, downloads the real `gpt2` weights from Hugging Face (first run only, then cached), and loads the deck's own palette so the plots read like the ones you just watched.

In [1]:
# Quiet, idempotent installs -- safe to re-run.
!pip install -q --upgrade transformers torch plotly

import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # baked notebook: no live widget to render into

import random
import torch
import plotly.graph_objects as go
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

# The deck's own palette, so the plots here read like the plots you just watched.
GOLD, BLUE, TEAL, RED, DIM = "#b07d22", "#2563a8", "#148a7a", "#c0392b", "#7f8c9a"

print(f"gpt2 loaded: {model.num_parameters():,} parameters, vocab {tokenizer.vocab_size:,}")


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Users\v-seydi\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


gpt2 loaded: 124,439,808 parameters, vocab 50,257


## Part 1 -- Build the Bench

**Problem.** "Set the temperature to 0.7" and "use nucleus sampling" are real API parameters, not just slide vocabulary -- every call to a hosted LLM picks a decoding rule, by default or on purpose. Slide 21 stopped at "the stack hands you four numbers, something still has to choose one word." This part builds the thing that chooses: a harness that runs the same GPT-2, on the same prompt, under every decoding rule from slides 22-24, and keeps every continuation so the rest of the lab has real text to measure instead of a description of it.

**Ingredients.** One fixed prompt and one fixed seed, so any difference between runs comes from the decoding rule, not from a different starting point.

In [2]:
PROMPT = "The best way to learn a new skill is"
MAX_NEW_TOKENS = 40

prompt_ids = tokenizer(PROMPT, return_tensors="pt")["input_ids"]
print(f"Prompt: {PROMPT!r}  ({prompt_ids.shape[1]} tokens)")

Prompt: 'The best way to learn a new skill is'  (9 tokens)


**Idea -- one function, one dict of `generate()` kwargs per strategy.** Every decoding rule from slides 22-24 is already implemented inside Hugging Face's `generate()` -- greedy is `do_sample=False`; temperature, top-k and nucleus are `temperature`, `top_k`, `top_p` under `do_sample=True`. So the design choice here is not to reimplement the softmax-with-temperature or the nucleus condition (slide 21's and slide 24's formulas already cover that), it is to hold everything else fixed -- same prompt, same seed reset before every call, same `max_new_tokens` -- so the *only* thing that varies between runs is the decoding rule itself, which is what makes the runs comparable in Part 2. The strategies swept: greedy; temperature in {0.3, 0.7, 1.0, 1.3} (slide 22's own knob); top-k in {3, 10, 50}; nucleus (top-p) in {0.7, 0.9, 0.95} (slide 24's own knob, including the 0.9 the quiz used).

In [3]:
def run_strategy(label, family, **gen_kwargs):
    """Reset the seed, generate, and return everything Part 2 will need to score."""
    torch.manual_seed(SEED)
    with torch.no_grad():
        out = model.generate(prompt_ids, max_new_tokens=MAX_NEW_TOKENS,
                              pad_token_id=tokenizer.eos_token_id, **gen_kwargs)
    full_ids = out[0]
    gen_ids = full_ids[prompt_ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    return {"label": label, "family": family, "full_ids": full_ids,
            "gen_ids": gen_ids, "text": text}


strategies = [("greedy", "greedy", dict(do_sample=False))]
for t in (0.3, 0.7, 1.0, 1.3):
    strategies.append((f"temp={t}", "temperature", dict(do_sample=True, temperature=t, top_k=0, top_p=1.0)))
for k in (3, 10, 50):
    strategies.append((f"top-k={k}", "top-k", dict(do_sample=True, top_k=k, temperature=1.0)))
for p in (0.7, 0.9, 0.95):
    strategies.append((f"top-p={p}", "nucleus", dict(do_sample=True, top_p=p, top_k=0, temperature=1.0)))

runs = [run_strategy(label, family, **kwargs) for label, family, kwargs in strategies]
print(f"{len(runs)} runs generated.")

11 runs generated.


In [4]:
for r in runs:
    print(f"{r['label']:>10}  {r['text'][:90]!r}")

    greedy  ' to learn it from someone who has done it before.\n\nThe best way to learn a new skill is to'
  temp=0.3  " to practice it. It's a great way to learn, but it's not the best way to learn.\n\nThe best "
  temp=0.7  ' to practice it a little.\n\nMoreover, you will gain some experience once you have mastered '
  temp=1.0  " to study it a little. It has more resources than science. Remember: once again, we can't "
  temp=1.3  ' to just ask a teacher. Experts know more about failed, inappropriate moves like hangover '
   top-k=3  " to practice it for a while.\n\nYou can also practice it for a few days, and then you'll lea"
  top-k=10  ' to study it and apply it in your own work.\n\nYou need a new job to get it. The best way to'
  top-k=50  " to study it a little. It has more resources than science. It's hard and it's hard. The mo"
 top-p=0.7  ' to study it and apply it to your own work. And to learn something from others, we can lea'
 top-p=0.9  ' to study it a little. It has mo

**Payoff.** The four numbers on slide 21 turned into eleven different endings for the exact same nine-token prompt, and the shape matches what slides 22-24 predicted in words. Greedy loops back onto the prompt itself almost immediately -- "to learn it from someone who has done it before... The best way to learn a new skill is to" -- the exact failure `greedy_trap` warned about on slide 23, not a paraphrase of it. At the other extreme, `temp=1.3` wanders into "Experts know more about failed, inappropriate moves like hangover," a sentence that parses but does not track. And three settings that look nothing alike on paper -- `temp=1.0`, `top-k=50`, `top-p=0.9` -- all open with the identical clause "to study it a little. It has more resources than science," because the fixed seed replays the same random draws and none of the three filters removes the token that draw lands on until several steps in. That is the real limitation of reading text by eye: two settings can look the same for a sentence and then diverge, or look different and still cost the same underneath. Part 2 turns "looks fine" into a number, so that stops being a coin flip.

## Part 2 -- Measure the Trade-off

**Problem.** Reading eleven continuations and judging them by eye does not scale, and it is not how the setting actually gets chosen in production. This part turns each run into two numbers -- a repetition score and a plausibility score -- so every setting becomes a point you can compare, and the trade-off slide 24 described in words ("temperature reshapes everything... top-p flips the question round") becomes a shape you can actually see, on your own runs.

**Idea -- repetition rate as the distinct-trigram ratio.** Slide 24's own citation, Holtzman et al.'s "The Curious Case of Neural Text Degeneration," diagnoses degeneration by looking for repeated phrases, not repeated single words (natural text repeats individual words constantly; it does not repeat three-word runs). So: slide the generated tokens through every 3-gram window, and score `1 - (distinct trigrams / total trigrams)`. A clean, non-looping continuation scores near 0; the greedy loop slide 23's `greedy_trap` warned about scores much higher.

In [5]:
def repetition_rate(token_ids):
    ids = token_ids.tolist()
    if len(ids) < 3:
        return 0.0
    trigrams = [tuple(ids[i:i + 3]) for i in range(len(ids) - 2)]
    return 1 - len(set(trigrams)) / len(trigrams)


for r in runs:
    r["repetition"] = repetition_rate(r["gen_ids"])

**Idea -- score every run under the same, unwarped model.** The obvious plausibility score is the log-probability `generate()` assigned each token -- but that number was computed *after* temperature or top-p reshaped the distribution (slide 21's own point), so a high-temperature run scored under its own warped distribution would look artificially confident about tokens the base model actually found unlikely: it would be grading its own homework. The fix is the one move slide 08 already taught for a different reason -- teacher forcing: run the full generated sequence back through the model in a single forward pass, at the model's raw T=1 distribution, and read off the log-probability it assigns to the tokens each strategy actually chose. That number is comparable across every strategy because it is scored by the same fixed yardstick every time.

In [6]:
def mean_logprob(full_ids, prompt_len):
    with torch.no_grad():
        logits = model(full_ids.unsqueeze(0)).logits[0]
    logprobs = torch.log_softmax(logits, dim=-1)
    gen_ids = full_ids[prompt_len:]
    token_logprobs = [logprobs[prompt_len - 1 + i, tok_id].item()
                       for i, tok_id in enumerate(gen_ids)]
    return sum(token_logprobs) / len(token_logprobs)


for r in runs:
    r["logprob"] = mean_logprob(r["full_ids"], prompt_ids.shape[1])

for r in runs:
    print(f"{r['label']:>10}  repetition={r['repetition']:.3f}  logprob={r['logprob']:.3f}")

    greedy  repetition=0.237  logprob=-0.776
  temp=0.3  repetition=0.289  logprob=-0.905
  temp=0.7  repetition=0.026  logprob=-2.330
  temp=1.0  repetition=0.000  logprob=-3.495
  temp=1.3  repetition=0.000  logprob=-5.842
   top-k=3  repetition=0.211  logprob=-1.494
  top-k=10  repetition=0.211  logprob=-1.424
  top-k=50  repetition=0.000  logprob=-2.852
 top-p=0.7  repetition=0.000  logprob=-1.871
 top-p=0.9  repetition=0.000  logprob=-3.080
top-p=0.95  repetition=0.000  logprob=-3.080


In [7]:
family_color = {"greedy": RED, "temperature": GOLD, "top-k": BLUE, "nucleus": TEAL}

fig = go.Figure()
for family in ("greedy", "temperature", "top-k", "nucleus"):
    pts = [r for r in runs if r["family"] == family]
    fig.add_trace(go.Scatter(
        x=[r["repetition"] for r in pts], y=[r["logprob"] for r in pts],
        mode="markers+text", text=[r["label"] for r in pts], textposition="top center",
        marker=dict(size=12, color=family_color[family]), name=family))
fig.update_layout(title="Repetition rate vs. mean log-probability, per decoding setting",
                   xaxis_title="repetition rate (lower = less looping)",
                   yaxis_title="mean token log-probability (higher = more plausible to the model)",
                   width=760, height=480)
fig.show()

**Payoff.** The scatter splits into two clusters, not a smooth line. `greedy`, `temp=0.3`, `top-k=3` and `top-k=10` sit together at repetition 0.21-0.29 and the best plausibility in the sweep (logprob -0.78 to -1.5): safe, but visibly looping, exactly the pocket `greedy_trap` occupies. Everything that opens the candidate set further -- `temp>=0.7`, `top-k=50`, every nucleus setting tried -- collapses repetition to 0.000-0.026 on this run, at a real and growing plausibility cost: -1.87 at the gentlest of them (`top-p=0.7`) down to -5.84 at `temp=1.3`, more than double the next-worst setting. That is the trade-off slide 24 described in words, now with your own numbers on both axes. The limitation is real too: this is one prompt and one seed, so a single point is one noisy draw from what a setting can do, not its expected behaviour -- `top-p=0.9` and `top-p=0.95` land on the exact same point here, which is a coincidence of this draw, not a guarantee that 0.9 and 0.95 always agree. The practical takeaway: temperature and top-p are not "quality" dials, they are a dial between two failure modes, boilerplate looping on one end and unmoored plausibility on the other, and an API's default sits somewhere on that line whether you chose it or not.

## Part 3 -- Few-shot, and Where It Stops

**Problem.** Slide 18 named it: GPT-3 performs a task from examples in the prompt with no gradient update at all -- "in-context learning." Nobody in the lecture told you how many examples that actually needs, and gpt2, at 124M parameters, is nowhere near GPT-3's scale. This part finds gpt2's own answer empirically: prepend a sentiment-labelling example, one at a time, and find the point where the model starts reliably naming the right label -- then find a phrasing where it stops.

**Ingredients.** A small pool of hand-written, unambiguous review/label pairs to draw the few-shot examples from, and a held-out set of test reviews the model never sees as an example.

In [8]:
shot_pool = [
    ("This movie was absolutely wonderful, I loved every second.", "positive"),
    ("I hated this film, it was a complete waste of time.", "negative"),
    ("The acting was superb and the story kept me hooked.", "positive"),
    ("Boring, predictable, and far too long.", "negative"),
]

test_cases = [
    ("The plot dragged but the visuals were stunning.", "positive"),
    ("A forgettable mess with no redeeming qualities.", "negative"),
    ("Best film I have seen all year, truly brilliant.", "positive"),
    ("Two hours I will never get back.", "negative"),
]


def build_prompt(k, review):
    lines = [f"Review: {r}\nSentiment: {label}" for r, label in shot_pool[:k]]
    lines.append(f"Review: {review}\nSentiment:")
    return "\n\n".join(lines)

**Idea -- greedy, on purpose, swept over shot count.** Greedy decoding is deterministic (Part 1's own point), so any change in accuracy as `k` grows is attributable purely to the number of examples in the prompt, not to sampling noise on top of it. For each `k` from 0 to 4, build the prompt, generate three tokens, and check whether the model's own words name the correct label.

In [9]:
def predict_label(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=3, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


accuracy_by_k = {}
for k in range(0, 5):
    correct = 0
    for review, label in test_cases:
        cont = predict_label(build_prompt(k, review)).lower()
        pred = "positive" if "positive" in cont else ("negative" if "negative" in cont else "neither")
        correct += int(pred == label)
    accuracy_by_k[k] = correct / len(test_cases)
    print(f"k={k}  accuracy={accuracy_by_k[k]:.2f}")

k=0  accuracy=0.00


k=1  accuracy=0.50


k=2  accuracy=0.75


k=3  accuracy=0.50


k=4  accuracy=0.50


**Idea -- find a phrasing that breaks the best k.** The shot-count sweep only tests reviews that state their sentiment directly. Negation ("I did not expect to enjoy this as much as I did") states the opposite surface words from the sentiment it actually carries, which is exactly the case where a model that is pattern-matching surface words rather than resolving meaning should fail. Run it at the k that scored highest above.

In [10]:
best_k = max(accuracy_by_k, key=accuracy_by_k.get)
breaking_case = "I did not expect to enjoy this as much as I did."
true_label = "positive"

cont = predict_label(build_prompt(best_k, breaking_case)).lower()
pred = "positive" if "positive" in cont else ("negative" if "negative" in cont else "neither")
print(f"best_k = {best_k}")
print(f"{breaking_case!r} -> predicted {pred!r}, true label {true_label!r}")

best_k = 2
'I did not expect to enjoy this as much as I did.' -> predicted 'negative', true label 'positive'


**Payoff.** Accuracy jumps from 0.00 at k=0 straight to 0.50 at k=1: at zero shots the model was not naming "positive" or "negative" at all, so that jump is the model learning the label VOCABULARY, not suddenly getting smarter about sentiment. It peaks at k=2 (0.75) -- the point this part set out to find -- and then it does not keep climbing: k=3 and k=4 both drop back to 0.50, unlike GPT-3's TriviaQA curve on slide 20, which only ever went up (64.3 to 68.0 to 71.2 percent). At 124M parameters, gpt2 does not have the capacity to keep using more examples productively; past the point where it has learned the format, additional shots add noise it cannot resolve. The breaking phrase confirms the mechanism: at the best k, "I did not expect to enjoy this as much as I did" -- genuinely positive -- comes back predicted negative, because the model is matching the surface negation cue ("did not"), not resolving what the sentence actually says. Limitation: four test reviews per k is a small sample, so 0.50 versus 0.75 is a difference of two correct answers, not a statistically solid gap -- a real evaluation would need dozens of held-out reviews per k. The takeaway: "how many examples" and "does more always help" are empirical questions specific to a model's own capacity, not points on a universal curve, and in-context learning on a small model can be shallow enough that a single negation breaks it even at its own best setting.

## Challenge -- Break It Deliberately: One Setting, Two Jobs

**Problem.** Part 2's scatter has a temptation built into it: pick the point that looks best overall and ship it everywhere. Will that hold up? Take two real jobs with opposite priorities -- a **code assistant** (a syntax error is a hard failure; boilerplate repetition is normal and fine) and a **short-story opener** (a repeated sentence is a visible flaw; being slightly less "expected" is the whole point) -- and check whether one default setting serves both, or breaks one of them.

**Ingredients.** One code-completion prompt and one story-opening prompt, run through the exact same bench Part 1 built.

In [11]:
CODE_PROMPT = "def is_prime(n):\n    if n < 2:\n        return False\n    for i in range(2,"
STORY_PROMPT = "The lighthouse keeper had not seen another ship in eleven years, until the morning"

jobs = {"code": CODE_PROMPT, "story": STORY_PROMPT}
challenge_settings = [("greedy", "greedy", dict(do_sample=False)),
                       ("temp=0.3", "temperature", dict(do_sample=True, temperature=0.3, top_k=0, top_p=1.0)),
                       ("temp=1.0", "temperature", dict(do_sample=True, temperature=1.0, top_k=0, top_p=1.0)),
                       ("top-p=0.9", "nucleus", dict(do_sample=True, top_p=0.9, top_k=0))]

**Idea -- reuse the bench unmodified, then score each job by the metric that job actually cares about.** `run_strategy`, `repetition_rate` and `mean_logprob` were built generic in Part 1-2, so the design choice here is not new code, it is what to do with the two numbers each job produces: for the code prompt, prioritise `logprob` (does the completion look like real code the model finds plausible, since a broken `for` loop is not a matter of taste); for the story prompt, prioritise low `repetition` (a lighthouse story that loops back on its own sentence has failed regardless of how "plausible" that loop is).

In [12]:
def score_job(prompt_text, job_label, setting_label, **gen_kwargs):
    ids = tokenizer(prompt_text, return_tensors="pt")["input_ids"]
    torch.manual_seed(SEED)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=30, pad_token_id=tokenizer.eos_token_id, **gen_kwargs)
    gen_ids = out[0][ids.shape[1]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    rep = repetition_rate(gen_ids)
    lp = mean_logprob(out[0], ids.shape[1])
    return {"job": job_label, "setting": setting_label, "text": text, "repetition": rep, "logprob": lp}


challenge_results = []
for job_name, job_prompt in jobs.items():
    for setting_label, family, gen_kwargs in challenge_settings:
        challenge_results.append(score_job(job_prompt, job_name, setting_label, **gen_kwargs))

for r in challenge_results:
    print(f"[{r['job']:>5}] {r['setting']:>10}  rep={r['repetition']:.3f}  logprob={r['logprob']:.3f}  {r['text'][:60]!r}")

[ code]     greedy  rep=0.214  logprob=-0.626  ' n):        return False\n    return True\n\nreturn False\n\ndef '
[ code]   temp=0.3  rep=0.393  logprob=-0.544  ' 1):        return True\n    if n > 2:        return False\n'
[ code]   temp=1.0  rep=0.036  logprob=-4.105  ' 1): \xa0 a = Request( False , \xa0di ) \xa0     we can now run get_i'
[ code]  top-p=0.9  rep=0.143  logprob=-2.340  ' 1):\n        return n-1+i+1\n\nAny other error here is >>>>>> '
[story]     greedy  rep=0.321  logprob=-1.652  ' of the 13th. He had been in the lighthouse for a few days, '
[story]   temp=0.3  rep=0.000  logprob=-1.850  ' of the first day of the month of July, when he saw the ship'
[story]   temp=1.0  rep=0.000  logprob=-4.063  ' of 29 January 903. He had more than failed to get his lante'
[story]  top-p=0.9  rep=0.000  logprob=-3.854  ' of 29 January 1942, when Charles Augustus Holmberg failed t'


**Payoff.** The two jobs disagree about which setting wins, using the numbers above. For the code prompt, `greedy` is strongest: valid Python, the best logprob of the four (-0.626), and the lowest repetition (0.214). Both "creative" settings break it outright -- `temp=1.0` drifts into dead syntax ("a = Request( False , di ) we can now run get_index"), `top-p=0.9` degrades into stray arrows ("Any other error here is >>>>>>") -- and the model's own logprob agrees, dropping to -4.105 and -2.340. For the story prompt `greedy` is the worst of the four: it is the only setting that visibly repeats itself (0.321 versus 0.000 everywhere else), landing right back on "he had been in the lighthouse for a few days, and had been in the lighthouse for a few days" -- the exact failure a story cannot afford. `temp=0.3` and `top-p=0.9` both read as fluent, non-repeating story openings, at a plausibility cost the story job does not need to care about. So: greedy, or a low temperature near 0.2-0.3, for the code assistant, because a syntax break is a hard failure and greedy nearly eliminates that risk here; something with real entropy, `temp=0.3` to `top-p=0.9`, for the story generator, because greedy's own boilerplate risk is the single worst score in the whole table for that job. Limitation: this is one prompt per job on a 124M-parameter model -- a production decision needs this same bench run over many prompts and a bigger model, since "syntax breaks under high temperature" is partly a capacity ceiling here, not only a temperature effect. The practical takeaway, stated plainly: there is no setting that is simply "the best," only a setting that is best for the number a specific job actually cannot compromise on.

## Reflection

You now own the full decoding toolkit: greedy, temperature, top-k, nucleus, how to score a run instead of eyeballing it, and where in-context learning starts and stops on a small model. The point of this cell is not a single right answer -- it is putting your own results into words, in your own voice.

In [13]:
# 1. Part 2's scatter put every setting on the same two axes. Greedy sits at
#    one extreme and the highest temperature sits at roughly the other. In
#    your own words, what is the ONE thing you are actually trading away
#    when you turn temperature up, and what are you buying with it?
answer_1 = """
"""

# 2. Slide 24 said top-k keeps a fixed COUNT of words and top-p keeps a fixed
#    AMOUNT of probability. Part 2 ran both. Using your own repetition and
#    logprob numbers, describe one situation where that difference actually
#    changed which setting you would pick.
answer_2 = """
"""

# 3. Part 3's accuracy-by-k curve did not necessarily climb smoothly the way
#    GPT-3's TriviaQA curve did on slide 20 (64.3 / 68.0 / 71.2). What does
#    gpt2's much smaller parameter count have to do with the shape of the
#    curve you actually got?
answer_3 = """
"""

# 4. The breaking phrasing in Part 3 used negation. Why would a model that
#    is pattern-matching surface words in the prompt, rather than resolving
#    what the sentence actually means, be specifically vulnerable to
#    negation -- and not, say, to a long sentence?
answer_4 = """
"""

# 5. The Challenge scored "code" and "story" by two DIFFERENT metrics on
#    purpose. If you had to pick one single number to optimise for every
#    job a production system might be asked to do, what would you lose by
#    doing that?
answer_5 = """
"""

---
*GPT: Predicting What Comes Next -- Bluevision-ai Academy*